# PrimeKG integrado más la red de compuestos del laboratorio

Este notebook arma el grafo de PrimeKG integrado unido con la red de compuestos del laboratorio
(`datos/out_b/`). Nada se carga en Neo4j: eso lo hace `../neo4j/cargar_grafo_laboratorio.py`.

**Entra D\***: un compuesto entra si es una droga del
listado de 1.801 drogas con `indication` en PrimeKG, o si tiene bioactividad positiva hacia algún gen
humano del grafo. Las reglas del cruce:

- Un nodo por compuesto, con su cluster como propiedad. No hay nodo por cluster.
- Droga a droga: las aristas del laboratorio entre clusters (similaridad de 0,81 a 0,99) pasan a todos
  los pares de compuestos de esos clusters. Los compuestos de un mismo cluster no se unen entre sí, y se
  descartan las aristas de un cluster consigo mismo.
- Droga a gen: solo actividades positivas, de UniProt a Entrez con el archivo de ID mapping humano, y
  solo hacia genes del grafo (sin los complejos de SIGNOR).
- Droga a enfermedad: `indication` de PrimeKG, con la misma regla que se usó para MONDO (al grupo BERT
  si el nodo de PrimeKG es un agrupamiento, a los CUIs si es un MONDO suelto).
- Quedan afuera los dominios y ortogrupos de proteínas, las actividades negativas y el año.

In [ ]:
import sys
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

CARPETA_CAPA = "datos"
CARPETA_ENTREGA = "datos/out_b"
CARPETA_INTEGRADO = "../datos/processed"

#las tres funciones de enfermedades salen de armar_grafo.py, el mismo cruce en script, asi las dos versiones no se separan
sys.path.insert(0, ".")
from armar_grafo import puente_enfermedades, indicaciones_de_primekg, aristas_droga_enfermedad, ARCHIVO_NODOS_PRIMEKG

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

#colores: azul para lo del grafo integrado, naranja para lo que suma el cruce
COLOR_INTEGRADO = "#2a78d6"
COLOR_NUEVO = "#eb6834"

## 1. Las tablas del cruce

Tamaño y primeras filas de cada tabla que usa el cruce. Las últimas dos, dominios (InterPro) y
ortogrupos (OrthoMCL), no entran al grafo y se muestran solo para ver qué trae la entrega. Las
indicaciones salen de las aristas `indication` de PrimeKG nativo, que lee `armar_grafo.py`.

In [ ]:
compuestos_lab = pd.read_csv(f"{CARPETA_ENTREGA}/04b_drogas_TDR.csv", dtype=str)
clusters_lab = pd.read_csv(f"{CARPETA_ENTREGA}/04b_tclus_TDR.csv", dtype=str)
aristas_lab = pd.read_csv(f"{CARPETA_ENTREGA}/04b_edges_fp_TDR.csv", dtype={"clusID1": str, "clusID2": str})
bioact_lab = pd.read_csv(f"{CARPETA_ENTREGA}/04b_bioact_TDR.csv", dtype=str)
interpro_lab = pd.read_csv(f"{CARPETA_ENTREGA}/04b_interpro_TDR.csv", dtype=str)
ortogrupos_lab = pd.read_csv(f"{CARPETA_ENTREGA}/04b_omcl_TDR.csv", dtype=str)
listado = pd.read_csv(f"{CARPETA_CAPA}/01_drogas_con_chembl_ccd_smiles.csv", dtype=str)
#accession humana a GeneID, del archivo de ID mapping humano de UniProt (columnas 1 y 3, sin cabecera)
mapeo_humano = pd.read_csv(f"{CARPETA_CAPA}/HUMAN_9606_idmapping_selected.tab.gz", sep="\t", header=None,
                           usecols=[0, 2], names=["accession", "geneids"], dtype=str)
nodos_integrado = pd.read_csv(f"{CARPETA_INTEGRADO}/merged_nodes.csv")
nodos_integrado["node_id"] = nodos_integrado.node_id.astype(str)
filas_integrado = pd.read_csv(f"{CARPETA_INTEGRADO}/merged_edges.csv")
filas_integrado["x_id"] = filas_integrado.x_id.astype(str)
filas_integrado["y_id"] = filas_integrado.y_id.astype(str)

for nombre, tabla in [("04b_drogas, los compuestos", compuestos_lab), ("04b_tclus, compuesto a cluster", clusters_lab),
                      ("04b_edges_fp, similaridad entre clusters", aristas_lab), ("04b_bioact, compuesto a proteina", bioact_lab),
                      ("01, el listado de drogas con indication", listado), ("ID mapping humano de UniProt", mapeo_humano),
                      ("merged_nodes, nodos del grafo integrado", nodos_integrado), ("merged_edges, aristas del grafo integrado", filas_integrado),
                      ("04b_interpro, dominios (no entra)", interpro_lab), ("04b_omcl, ortogrupos (no entra)", ortogrupos_lab)]:
    print(f"{nombre}: {tabla.shape[0]} filas, {tabla.shape[1]} columnas")
    display(tabla.head(3))

## 2. El cruce

Todo en una celda, en este orden:

1. El grafo integrado, con cada arista una vez (el CSV guarda cada una en los dos sentidos).
2. Las drogas del listado a compuestos, por ChEMBL o por CCD.
3. Los pares compuesto-gen: positivas, accession a GeneID con el archivo humano, solo genes del grafo.
4. D\*: los compuestos del listado más los que tienen algún par compuesto-gen.
5. Un nodo por compuesto de D\*, con `node_index` consecutivo desde el máximo del grafo.
6. Droga a enfermedad: cada droga pasa sus indicaciones a todos sus compuestos.
7. Droga a droga: cada arista entre dos clusters distintos pasa a todos los pares de compuestos de D\*
   de esos clusters, con su peso.
8. Las aristas nuevas con las columnas del grafo integrado, más `weight`.
9. Se escriben `nodos.csv`, `aristas.csv` y `drogas_detalle.csv`.

In [ ]:
#1. el grafo integrado, con cada arista una vez
#el CSV guarda cada arista dos veces, una en cada sentido. antes de quedarse con una se comprueba que toda
#fila tenga su inversa, porque si faltara alguna el filtro de abajo perderia esa arista sin avisar
pares_integrado = set(zip(filas_integrado.x_index, filas_integrado.y_index))
assert all((b, a) in pares_integrado for a, b in pares_integrado), "hay filas sin su inversa"
#un autobucle es su propia inversa y el filtro x menor que y tambien lo perderia
assert (filas_integrado.x_index != filas_integrado.y_index).all(), "merged_edges.csv trae autobucles"
COLUMNAS_ARISTA = ["x_index", "y_index", "x_id", "y_id", "x_type", "y_type", "edge_type", "edge_source"]
#de cada par de filas queda la que tiene el indice menor en x
aristas_integrado = filas_integrado[filas_integrado.x_index < filas_integrado.y_index][COLUMNAS_ARISTA].reset_index(drop=True)
nodos_base = nodos_integrado[["node_index", "node_id", "node_name", "node_type", "node_source"]]
#pasa de un id del grafo (el Entrez de un gen, el CUI de una enfermedad) a su node_index
indice_por_node_id = nodos_base.set_index("node_id").node_index
#los genes a los que puede llegar una diana. los complejos de SIGNOR son nodos gene_protein pero no tienen
#Entrez, asi que ninguna accession de UniProt puede caer en ellos
genes_integrado = nodos_base[nodos_base.node_type == "gene_protein"].node_id
genes_del_grafo = set(genes_integrado[~genes_integrado.str.startswith("SIGNOR")])

#2. drogas del listado a compuestos, por el id de ChEMBL o por el codigo CCD
#cada merge da los pares (droga, compuesto) que comparten ese id. los nulos se sacan de la tabla del
#laboratorio para que un id vacio del listado no se empareje con un id vacio de la red
por_chembl = listado.merge(compuestos_lab[["chembl_id", "CID"]].dropna(), on="chembl_id")
por_ccd = listado.merge(compuestos_lab[["biolip_id", "CID"]].dropna(), left_on="ccd_id_biolip", right_on="biolip_id")
#una droga puede cruzar con el mismo compuesto por los dos ids, de ahi el drop_duplicates
droga_compuesto = pd.concat([por_chembl, por_ccd])[["drugbank_id", "CID"]].drop_duplicates().reset_index(drop=True)

#3. pares compuesto-gen, solo con actividad positiva y solo hacia genes del grafo, con el origen de cada par
#el archivo de UniProt trae los GeneID de cada accession en un solo campo, separados por punto y coma. con explode
#queda una fila por (accession, GeneID), y dropna saca las accessions que no tienen GeneID
accession_gen = mapeo_humano.assign(gene_id=mapeo_humano.geneids.str.split("; ")).explode("gene_id").dropna()[["accession", "gene_id"]]
positivas = bioact_lab[bioact_lab.activity_tag == "positive"]
#las fusiones de cristalografia de BioLiP traen varias accessions en el mismo target_id, separadas por coma.
#se parten para mapear cada una por su cuenta
positivas = positivas.assign(accession=positivas.target_id.str.split(",")).explode("accession")
#el merge deja afuera las accessions que no estan en el archivo humano, que son las dianas de otras especies
compuesto_gen = positivas.merge(accession_gen, on="accession").rename(columns={"drug_id": "CID"})
#de los GeneID humanos quedan los que son genes del grafo
compuesto_gen = compuesto_gen[compuesto_gen.gene_id.isin(genes_del_grafo)]
#un par puede llegar por varias filas. si esas filas traen origenes distintos, o alguna ya dice both, el par es both
compuesto_gen = compuesto_gen.groupby(["CID", "gene_id"]).origen.agg(
    lambda valores: "both" if len(set(valores)) > 1 or "both" in set(valores) else valores.iloc[0]).reset_index()

#4. D*, entra un compuesto que sea droga del listado o que tenga algun gen del grafo como diana
compuestos_del_listado = set(droga_compuesto.CID)
compuestos_con_gen = set(compuesto_gen.CID)
#el CID es texto, se ordena como numero para que los node_index nuevos sigan el orden del laboratorio
d_estrella = sorted(compuestos_del_listado | compuestos_con_gen, key=int)

#5. un nodo por compuesto
#04b_drogas tiene varias filas por CID, una por cada id externo que el laboratorio junto en ese compuesto.
#el peso es el mismo en todas, se toma de la primera
primera_fila = compuestos_lab.drop_duplicates("CID").set_index("CID").loc[d_estrella]
#de ChEMBL y CCD se guardan todos los ids distintos del compuesto, unidos con |
chembl_por_cid = (compuestos_lab.dropna(subset=["chembl_id"]).drop_duplicates(["CID", "chembl_id"])
                  .sort_values("chembl_id").groupby("CID").chembl_id.agg("|".join))
ccd_por_cid = (compuestos_lab.dropna(subset=["biolip_id"]).drop_duplicates(["CID", "biolip_id"])
               .sort_values("biolip_id").groupby("CID").biolip_id.agg("|".join))
#las drogas del listado de cada compuesto, con los DrugBank y los nombres unidos. casi siempre es una sola,
#las excepciones son las sales de citrato y de lactato, que comparten compuesto
compuesto_con_droga = droga_compuesto.merge(listado[["drugbank_id", "droga"]], on="drugbank_id").sort_values("drugbank_id")
drogas_del_compuesto = compuesto_con_droga.groupby("CID").agg(drugbank_id=("drugbank_id", "|".join), nombre=("droga", " | ".join))
#la tabla queda indexada por CID, y cada columna se alinea por ese indice
compuestos_nuevos = pd.DataFrame({
    #los node_index nuevos arrancan donde termina el grafo integrado
    "node_index": range(nodos_base.node_index.max() + 1, nodos_base.node_index.max() + 1 + len(d_estrella)),
    "node_id": "CID_" + primera_fila.index,
    #el nombre de la droga si es del listado, si no el primer ChEMBL, y si no tiene ninguno el id del laboratorio
    "node_name": drogas_del_compuesto.nombre.reindex(d_estrella).fillna(primera_fila.chembl_id).fillna(primera_fila.compound_id),
    #reindex deja vacio el DrugBank de los compuestos que no son drogas del listado
    "drugbank_id": drogas_del_compuesto.drugbank_id.reindex(d_estrella),
    "chembl_id": chembl_por_cid.reindex(d_estrella),
    "ccd": ccd_por_cid.reindex(d_estrella),
    #los compuestos que no estan en 04b_tclus quedan sin cluster
    "cluster_id": clusters_lab.set_index("drug").cluster_id.reindex(d_estrella),
    "molweight": primera_fila.molweight,
}, index=d_estrella)
nodos_droga = compuestos_nuevos[["node_index", "node_id", "node_name"]].assign(node_type="drug", node_source="laboratorio")

#6. droga a enfermedad
#las tres funciones de armar_grafo.py llevan las indication de PrimeKG a los nodos Disease del grafo
#(grupo BERT o CUIs, segun el nodo de PrimeKG). del lado de la droga devuelven el DrugBank
nodos_primekg = pd.read_csv(ARCHIVO_NODOS_PRIMEKG, dtype={"node_id": str})
indicaciones_por_drugbank = aristas_droga_enfermedad(pd.DataFrame({"id": droga_compuesto.drugbank_id.unique()}),
                                                     puente_enfermedades(nodos_integrado, nodos_primekg),
                                                     indicaciones_de_primekg(nodos_primekg))
#el merge con droga_compuesto pasa cada indicacion a todos los compuestos de la droga
droga_enfermedad = indicaciones_por_drugbank.merge(droga_compuesto, left_on="drug_id", right_on="drugbank_id")[["CID", "disease_id"]].drop_duplicates()
compuestos_nuevos["tiene_indicacion"] = compuestos_nuevos.index.isin(droga_enfermedad.CID)

#7. droga a droga
#cada compuesto de D* con su cluster. los que no tienen cluster no pueden recibir aristas quimicas
miembros = compuestos_nuevos[["cluster_id"]].rename_axis("CID").reset_index().dropna()
#se descartan las aristas de un cluster consigo mismo. el primer merge trae los compuestos de D* del primer
#cluster y el segundo los del otro, asi que cada arista entre clusters se repite para todos los pares de
#compuestos. si un cluster no tiene compuestos en D*, el merge no encuentra pareja y la arista se cae
droga_droga = (aristas_lab[aristas_lab.clusID1 != aristas_lab.clusID2]
               .merge(miembros.rename(columns={"CID": "CID_a"}), left_on="clusID1", right_on="cluster_id")
               .merge(miembros.rename(columns={"CID": "CID_b"}), left_on="clusID2", right_on="cluster_id"))
#el laboratorio trae algunos pesos como 0.8099999 o como promedios largos. se redondean a 4 decimales
droga_droga["weight"] = droga_droga.weight.round(4)

#8. las aristas nuevas, con la droga siempre en x
nodos = pd.concat([nodos_base, nodos_droga], ignore_index=True)
#da el node_id y el tipo de cada nodo a partir de su node_index
por_indice = nodos.set_index("node_index")
#los tres tipos de arista se arman con la misma forma y se juntan. weight queda vacio salvo en droga a droga,
#y origen salvo en droga a gen
pares_nuevos = pd.concat([
    pd.DataFrame({"x_index": droga_enfermedad.CID.map(compuestos_nuevos.node_index), "y_index": droga_enfermedad.disease_id.map(indice_por_node_id),
                  "edge_type": "drug_disease", "edge_source": "primekg"}),
    pd.DataFrame({"x_index": compuesto_gen.CID.map(compuestos_nuevos.node_index), "y_index": compuesto_gen.gene_id.map(indice_por_node_id),
                  "edge_type": "drug_gene", "edge_source": "laboratorio", "origen": compuesto_gen.origen}),
    pd.DataFrame({"x_index": droga_droga.CID_a.map(compuestos_nuevos.node_index), "y_index": droga_droga.CID_b.map(compuestos_nuevos.node_index),
                  "edge_type": "drug_drug", "edge_source": "laboratorio", "weight": droga_droga.weight}),
], ignore_index=True)
#los ids y los tipos de los dos extremos se completan de una vez para los tres tipos
aristas_nuevas = pares_nuevos.assign(x_id=pares_nuevos.x_index.map(por_indice.node_id), y_id=pares_nuevos.y_index.map(por_indice.node_id),
                                     x_type=pares_nuevos.x_index.map(por_indice.node_type), y_type=pares_nuevos.y_index.map(por_indice.node_type))
#las columnas del grafo integrado, en el mismo orden, mas weight y origen
aristas_nuevas = aristas_nuevas[COLUMNAS_ARISTA + ["weight", "origen"]]
#una tabla por tipo, para las verificaciones
aristas_droga_enfermedad_nuevas = aristas_nuevas[aristas_nuevas.edge_type == "drug_disease"]
aristas_droga_gen = aristas_nuevas[aristas_nuevas.edge_type == "drug_gene"]
aristas_droga_droga = aristas_nuevas[aristas_nuevas.edge_type == "drug_drug"]

#9. escribir
#lo del grafo integrado va primero y sin cambios, las verificaciones lo comparan por posicion
aristas = pd.concat([aristas_integrado, aristas_nuevas], ignore_index=True)
nodos.to_csv("nodos.csv", index=False)
aristas.to_csv("aristas.csv", index=False)
compuestos_nuevos[["node_id", "drugbank_id", "chembl_id", "ccd", "cluster_id", "molweight", "tiene_indicacion"]].to_csv("drogas_detalle.csv", index=False)

print("D*:", len(d_estrella), "compuestos | del listado:", len(compuestos_del_listado),
      "| por bioactividad:", len(compuestos_con_gen), "| las dos cosas:", len(compuestos_del_listado & compuestos_con_gen))
print("aristas nuevas:", aristas_nuevas.edge_type.value_counts().to_dict())
print("escritos", len(nodos), "nodos y", len(aristas), "aristas")

## 3. Verificaciones

Si algo no cierra, esta celda falla. Comprueba que el grafo integrado quede intacto, que las aristas
nuevas tengan extremos y tipos válidos, sin repetidas ni autobucles, y que al pasar las aristas del
laboratorio de clusters a compuestos no se pierda ningún par.

Un detalle del cruce que conviene tener presente: algunas drogas comparten compuesto. Las sales de
citrato (de potasio, de sodio y de litio) quedan en un solo compuesto, y lo mismo el lactato de calcio y
el de magnesio, porque el laboratorio guarda la estructura sin el contraión. El compuesto junta las
indicaciones de todas.

In [ ]:
#el grafo integrado queda intacto
assert nodos.node_index.is_unique and nodos.node_id.is_unique, "node_index o node_id repetido"
assert nodos.iloc[:len(nodos_base)].equals(nodos_base.reset_index(drop=True)), "cambiaron los nodos del grafo integrado"
assert aristas.iloc[:len(aristas_integrado)][COLUMNAS_ARISTA].equals(aristas_integrado), "cambiaron las aristas del grafo integrado"

#extremos, tipos y repetidos
assert aristas.x_index.isin(por_indice.index).all() and aristas.y_index.isin(por_indice.index).all(), "arista con extremo inexistente"
assert aristas_nuevas[COLUMNAS_ARISTA].notna().all().all(), "arista nueva con algun campo vacio"
assert (aristas.x_index != aristas.y_index).all(), "autobucle"
assert not pd.DataFrame(np.sort(aristas[["x_index", "y_index"]].values, axis=1)).duplicated().any(), "arista repetida"
assert set(aristas_nuevas.x_type) == {"drug"}, "arista nueva sin droga en x"
assert set(aristas_droga_enfermedad_nuevas.y_type) == {"disease"} and set(aristas_droga_gen.y_type) == {"gene_protein"}, "arista nueva con el tipo equivocado en y"
assert aristas_droga_gen.origen.isin(["chembl", "biolip", "both"]).all(), "origen desconocido"

#droga a droga: pesos, ninguna arista dentro de un cluster, y ningun par perdido al replicar
cluster_por_node_id = compuestos_nuevos.set_index("node_id").cluster_id
assert aristas_droga_droga.weight.between(0.81, 0.99).all(), "peso fuera de rango"
assert (aristas_droga_droga.x_id.map(cluster_por_node_id) != aristas_droga_droga.y_id.map(cluster_por_node_id)).all(), "arista dentro de un cluster"
miembros_por_cluster = miembros.cluster_id.value_counts()
entre_clusters = aristas_lab[aristas_lab.clusID1 != aristas_lab.clusID2]
esperadas = (entre_clusters.clusID1.map(miembros_por_cluster) * entre_clusters.clusID2.map(miembros_por_cluster)).sum()
assert len(aristas_droga_droga) == esperadas, "faltan pares al replicar"

print("verificaciones ok")

## 4. Conexiones intra e inter capa

Las capas son los cuatro tipos de nodo: genes, enfermedades, pathways y drogas. Una arista intra capa une
dos nodos del mismo tipo (el interactoma, la jerarquía de enfermedades, la similaridad química) y una
inter capa une tipos distintos. La tabla tiene una fila por tipo de arista.

In [ ]:
CAPA = {"gene_protein": "gen", "disease": "enfermedad", "pathway": "pathway", "drug": "droga"}
ORDEN_CAPAS = ["gen", "enfermedad", "pathway", "droga"]
aristas["capa_x"] = aristas.x_type.map(CAPA)
aristas["capa_y"] = aristas.y_type.map(CAPA)
aristas["procedencia"] = np.where(aristas.index < len(aristas_integrado), "grafo integrado", "nuevo")
por_tipo = aristas.groupby(["edge_type", "capa_x", "capa_y", "procedencia"]).size().rename("aristas").reset_index()
por_tipo["conexion"] = np.where(por_tipo.capa_x == por_tipo.capa_y, "intra capa", "inter capa")
por_tipo = por_tipo.sort_values(["conexion", "aristas"], ascending=[False, False]).reset_index(drop=True)
por_tipo

In [ ]:
#una barra por tipo de arista, separadas en intra e inter capa, con el valor exacto al final de cada barra
fig, ejes = plt.subplots(1, 2, figsize=(13, 3.6), sharex=True)
for eje, conexion in zip(ejes, ["intra capa", "inter capa"]):
    filas = por_tipo[por_tipo.conexion == conexion].iloc[::-1]
    etiquetas = filas.edge_type + " (" + filas.capa_x + ", " + filas.capa_y + ")"
    colores = np.where(filas.procedencia == "nuevo", COLOR_NUEVO, COLOR_INTEGRADO)
    eje.barh(etiquetas, filas.aristas / 1000, color=colores, height=0.6)
    for posicion, valor in enumerate(filas.aristas):
        eje.text(valor / 1000, posicion, f" {valor:,}".replace(",", "."), va="center", fontsize=9, color="#52514e")
    eje.set_title(f"Aristas {conexion}")
    eje.spines[["top", "right"]].set_visible(False)
    eje.grid(axis="x", color="#e5e5e2", linewidth=0.8)
    eje.set_axisbelow(True)
for eje in ejes:
    eje.set_xlabel("miles de aristas")
ejes[1].set_xlim(0, por_tipo.aristas.max() / 1000 * 1.18)
fig.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=COLOR_INTEGRADO), plt.Rectangle((0, 0), 1, 1, color=COLOR_NUEVO)],
           labels=["del grafo integrado", "nuevas del cruce"], loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.06))
plt.tight_layout()
plt.show()

A la izquierda, cuántas aristas hay entre cada par de capas: la diagonal son las intra capa. A la
derecha, qué fracción de los nodos de cada capa (filas) tiene al menos un vecino en cada capa
(columnas). Entre paréntesis, los nodos de cada capa.

In [ ]:
#aristas entre cada par de capas, la diagonal son las intra capa. la matriz es simetrica
entre_capas = pd.crosstab(aristas.capa_x, aristas.capa_y).reindex(index=ORDEN_CAPAS, columns=ORDEN_CAPAS, fill_value=0)
entre_capas = entre_capas + entre_capas.T - np.diag(np.diag(entre_capas))

#para cada capa, que fraccion de sus nodos tiene al menos un vecino en cada capa
extremos = pd.concat([pd.DataFrame({"nodo": aristas.x_index, "capa_vecino": aristas.capa_y}),
                      pd.DataFrame({"nodo": aristas.y_index, "capa_vecino": aristas.capa_x})]).drop_duplicates()
extremos["capa_nodo"] = extremos.nodo.map(nodos.set_index("node_index").node_type.map(CAPA))
nodos_por_capa = nodos.node_type.map(CAPA).value_counts()
cobertura = (extremos.groupby(["capa_nodo", "capa_vecino"]).size().unstack(fill_value=0)
             .reindex(index=ORDEN_CAPAS, columns=ORDEN_CAPAS, fill_value=0).div(nodos_por_capa[ORDEN_CAPAS], axis=0) * 100)

fig, (izq, der) = plt.subplots(1, 2, figsize=(13, 4.8))
imagen = izq.imshow(entre_capas.replace(0, np.nan), cmap="Blues", norm=LogNorm(vmin=1e3, vmax=entre_capas.values.max()))
for (fila, columna), valor in np.ndenumerate(entre_capas.values):
    izq.text(columna, fila, f"{valor:,}".replace(",", "."), ha="center", va="center", fontsize=9,
             color="white" if valor > 3e4 else "#0b0b0b")
izq.set_title("Aristas entre capas (escala log)")
fig.colorbar(imagen, ax=izq, shrink=0.8)

imagen = der.imshow(cobertura, cmap="Blues", vmin=0, vmax=100)
for (fila, columna), valor in np.ndenumerate(cobertura.values):
    der.text(columna, fila, f"{valor:.0f} %", ha="center", va="center", fontsize=9, color="white" if valor > 60 else "#0b0b0b")
der.set_title("Nodos de la fila con algún vecino en la columna")
fig.colorbar(imagen, ax=der, shrink=0.8, label="% de los nodos de la fila")

for eje, etiqueta_x, etiqueta_y in [(izq, "capa", "capa"), (der, "capa del vecino", "")]:
    eje.set_xlabel(etiqueta_x)
    eje.set_ylabel(etiqueta_y)
    eje.set_xticks(range(4), ORDEN_CAPAS)
    eje.set_yticks(range(4), [f"{capa} ({nodos_por_capa[capa]:,})".replace(",", ".") for capa in ORDEN_CAPAS])
plt.tight_layout()
plt.show()

## Lectura

**Qué quedó.** El grafo tiene 119.706 nodos y 468.052 aristas. Lo del grafo integrado entra intacto
y se suman 83.867 compuestos: 1.481 son drogas del listado y 82.923 tienen bioactividad positiva hacia
algún gen del grafo, con 537 que cumplen las dos cosas.

**Las drogas pasan a ser la capa más grande y se conectan sobre todo con genes.** Son el 70 % de los
nodos. Droga a gen, con 131.973 aristas, es ahora el tipo de arista más numeroso del grafo, y el 99 %
de los compuestos tiene algún gen como vecino. Del otro lado, solo el 18 % de los genes tiene alguna
droga: las dianas se concentran en una parte del genoma.

**La conexión con enfermedades es la más flaca.** Droga a enfermedad tiene 8.496 aristas, y solo el
2 % de los compuestos tiene alguna enfermedad como vecina, porque la indicación la tienen solo las
drogas del listado. Visto desde las enfermedades, el 7 % tiene alguna droga indicada.

**El plano de drogas cubre menos de la mitad de los compuestos.** Las 72.557 aristas droga a droga son
el segundo plano intra capa después del interactoma, pero el 56 % de los compuestos no tiene ningún
vecino químico con el corte de 0,81.

**Los pathways solo tocan genes.** No hay aristas de pathways con enfermedades ni con drogas: una droga
llega a un pathway solo a través de sus genes.